# Homework 2 of LLM Zoomcamp class

## Q1. Embedding a query
### Embed the following query:

#### How does approximate nearest neighbor search work?

### The embedder returns a vector of 384 numbers. What's the first value (v[0])?


In [2]:
from embedder import Embedder

emb_model = Embedder()
v = emb_model.encode("How does approximate nearest neighbor search work?")
v[0]

np.float64(-0.02058200593003704)

### Answer: -0.02

## Loading the data

In [3]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

## Q2. Cosine similarity
### The embedder returns normalized vectors, so the dot product between two of them is their cosine similarity.

Take the page 02-vector-search/lessons/07-sqlitesearch-vector.md, embed its content, and compute the cosine similarity with the query vector from Q1. What do you get?

In [7]:
for doc in documents:
    if doc['filename'] == '02-vector-search/lessons/07-sqlitesearch-vector.md':
        v2 = emb_model.encode(doc['content'])

# Compute dot product similarity between the two vectors
cos_sim = sum(v * v2)
print(f"Cosine similarity between the query and the document: {cos_sim}")

Cosine similarity between the query and the document: 0.3610702906244368


### Answer: 0.37

## Q3. Chunking and search by hand

In [13]:
# A full page covers several topics, which waters down its embedding.
# We chunk the pages the same way as in homework 1:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

chunks[:5]

[{'start': 0,
  'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour ph

In [20]:
X = emb_model.encode_batch([chunk['content'] for chunk in chunks])

In [23]:
scores = X.dot(v)
max_score_chunk = chunks[scores.argmax()]
max_score_chunk

{'start': 1000,
 'content': 'rch. We score\nthe query against every document and pick the top ones. It always finds\nthe true top matches, but it pays for that by touching everything.\n\nApproximate nearest neighbor (ANN) search takes a shortcut. Instead of\ncomparing against everything, it first narrows down to a region of\nlikely matches. Then it scores only within that region. It may miss the\nabsolute best match, but the results are still good and it\'s much\nfaster.\n\n```text\nNN (exact):    compare query against ALL documents -> top 5\nANN (approx):  narrow down to a region -> compare within region -> top 5\n```\n\n## sqlitesearch\n\nsqlitesearch is the persistent sibling of minsearch, and it solves both\nproblems at once.\n\nWe already used it in module 1 for persistent text search. It also does\nvector search through its `VectorSearchIndex` class. It stores vectors\nin SQLite, a real on-disk database, and uses ANN strategies for\nretrieval. Because the data lives on disk, one 

### Answer: 02-vector-search/lessons/07-sqlitesearch-vector.md

## Q4. Vector search with minsearch

We've done vector search by hand, which is good for learning, but it's not
what we do in practice. In practice we use libraries.

Let's use `VectorSearch` from minsearch and run a search for the following
query:

> What metric do we use to evaluate a search engine?

Which file is the `filename` of the first result?

In [32]:
from minsearch import VectorSearch

vs = VectorSearch()
vs.fit(X, chunks)
query_vector = emb_model.encode("What metric do we use to evaluate a search engine?")
vs.search(query_vector, num_results=5)

[{'start': 0,
  'content': "# Search Evaluation Metrics\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=TuirMy3Pdbk&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we computed relevance lists for search results.\nWe can turn those lists into metrics.\n\n## Hit Rate\n\nHit Rate (also called Recall@k) measures the fraction of queries where\nthe correct document appears anywhere in the results:\n\n```python\nexample = [\n    [1, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 1, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n]\n```\n\nEach line is one query. If a line contains `1`, search found the\ncorrect document somewhere in the top 5 results. If the line contains\nonly zeros, search did not find the correct document.\n\nIn our set

### Answer: 04-evaluation/lessons/05-search-metrics.md

## Q5. Text search vs vector search

Vector search matches by meaning, keyword search by exact words.

Let's compare them. Index
the same chunks with `Index` from minsearch. Use `content` as a
text field.

Run both searches for this query:

> How do I store vectors in PostgreSQL?

Take the top 5 results from each method. Which file shows up in the
vector results but not in the text results?

* `02-vector-search/lessons/01-intro.md`
* `02-vector-search/lessons/02-embeddings.md`
* `02-vector-search/lessons/08-pgvector.md`
* `03-orchestration/lessons/05-rag.md`


In [38]:
from minsearch import Index

index = Index(text_fields=["content"])
index.fit(chunks)

index_results = index.search("How do I store vectors in PostgreSQL?", num_results=5)
print("Index results:")
print([result['filename'] for result in index_results])

vs_results = vs.search(emb_model.encode("How do I store vectors in PostgreSQL?"), num_results=5)
print("Vector search results:")
print([result['filename'] for result in vs_results])

Index results:
['02-vector-search/lessons/02-embeddings.md', '03-orchestration/lessons/05-rag.md', '02-vector-search/lessons/01-intro.md', '03-orchestration/lessons/05-rag.md', '02-vector-search/lessons/01-intro.md']
Vector search results:
['02-vector-search/lessons/08-pgvector.md', '02-vector-search/lessons/08-pgvector.md', '03-orchestration/lessons/05-rag.md', '02-vector-search/lessons/08-pgvector.md', '02-vector-search/lessons/08-pgvector.md']


### Answer: 02-vector-search/lessons/08-pgvector.md

## Q6. Hybrid search

Both vector and text search have their strengths and weaknesses. Vector
search matches by meaning, so it finds relevant pages even when they use
words different from the query. But it can miss exact terms like names,
codes, or rare keywords. Text search is the opposite: it nails exact words
but misses paraphrases and synonyms.

We don't have to pick one or the other - we can use both and merge their
results. This approach is called "hybrid search".

Each search produces its own ranked list, so we need a way to combine them
into one. In this homework we use Reciprocal Rank Fusion (RRF). It ignores
the raw scores from each method, which live on different scales and aren't
directly comparable. Instead, it looks only at the position of each
document in each list.

Every document scores by its position (`rank`, starting at 0) in each
list, and we sum the scores across lists with a constant `k = 60`:

```text
RRF(d) = sum over lists of  1 / (k + rank(d))
```

"Sum over lists" means we go through every ranked list and, for each list
where the document appears, add its `1 / (k + rank)` contribution. A
document found by both searches collects a score from each list, while one
found by only a single search collects just one.

The constant `k` controls how much the exact rank matters. A larger `k`
flattens the gap between positions, so the difference between rank 0 and
rank 5 counts for less. A smaller `k` does the opposite: it sharpens that
gap, so being at the top of a list matters much more.

The value 60 comes from the original RRF paper and is the usual default.
You rarely need to tune it. Lower it when only the top results matter.
Raise it to reward documents that appear across many lists, even when they
never quite reach the top.

A document that ranks well in both lists ends up higher than one that's
only strong in a single list.

In [39]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

Now run the query `"How do I give the model access to tools?"`
with vector and text search and fuse the results with `rrf`:


In [43]:
index_results = index.search("How do I give the model access to tools?", num_results=5)
print("Index results:")
print([result['filename'] for result in index_results])

vs_results = vs.search(emb_model.encode("How do I give the model access to tools?"), num_results=5)
print("Vector search results:")
print([result['filename'] for result in vs_results])

Index results:
['01-agentic-rag/lessons/14-agentic-loop.md', '01-agentic-rag/lessons/13-function-calling.md', '01-agentic-rag/lessons/13-function-calling.md', '01-agentic-rag/lessons/13-function-calling.md', '04-evaluation/lessons/02-ground-truth.md']
Vector search results:
['01-agentic-rag/lessons/01-intro.md', '04-evaluation/lessons/02-ground-truth.md', '01-agentic-rag/lessons/16-other-frameworks.md', '01-agentic-rag/lessons/15-frameworks.md', '01-agentic-rag/lessons/13-function-calling.md']


In [44]:
results = rrf([vs_results, index_results])
print([resultp['filename'] for resultp in results])

['01-agentic-rag/lessons/13-function-calling.md', '01-agentic-rag/lessons/01-intro.md', '01-agentic-rag/lessons/14-agentic-loop.md', '04-evaluation/lessons/02-ground-truth.md', '01-agentic-rag/lessons/16-other-frameworks.md']


Which file is ranked first after RRF?

* `01-agentic-rag/lessons/01-intro.md`
* `01-agentic-rag/lessons/13-function-calling.md`
* `01-agentic-rag/lessons/14-agentic-loop.md`
* `01-agentic-rag/lessons/16-other-frameworks.md`

Notice that this file isn't first in either search on its own - it wins
because it ranks high in both.


### Answer: 01-agentic-rag/lessons/13-function-calling.md